# Reflection Pattern

The first pattern we are going to implement is the **reflection pattern**. 

---

<img src="../img/reflection_pattern.png" alt="Alt text" width="600"/>

---

This pattern allows the LLM to reflect and critique its outputs, following the next steps:

1. The LLM **generates** a candidate output. If you look at the diagram above, it happens inside the **"Generate"** box.
2. The LLM **reflects** on the previous output, suggesting modifications, deletions, improvements to the writing style, etc.
3. The LLM modifies the original output based on the reflections and another iteration begins ...

**Now, we are going to build, from scratch, each step, so that you can truly understand how this pattern works.**

## Generation Step

The first thing we need to consider is:

> What do we want to generate? A poem? An essay? Python code?

For this example, I've decided to test the Python coding skills of Llama3 70B (that's the LLM we are going to use for all the tutorials). In particular, we are going to ask our LLM to code a famous sorting algorithm: **Merge Sort**. 

---

<img src="../img/mergesort.png" alt="Alt text" width="500"/>

### Groq Client and relevant imports

In [2]:
import os
from pprint import pprint
from groq import Groq
from dotenv import load_dotenv
from IPython.display import display_markdown

# Remember to load the environment variables. You should have the Groq API Key in there :)
load_dotenv()

client = Groq()

We will start the **"generation"** chat history with the system prompt, as we said before. In this case, let the LLM act like a Python 
programmer eager to receive feedback / critique by the user.

In [4]:
generation_chat_history = [
    {
        "role": "system",
        "content": "You are a Python programmer tasked with generating high quality Python code."
        "Your task is to Generate the best content possible for the user's request. If the user provides critique," 
        "respond with a revised version of your previous attempt."
    }
]

Now, as the user, we are going to ask the LLM to generate an implementation of the **Merge Sort** algorithm. Just add a new message with the **user** role to the chat history.

In [5]:
generation_chat_history.append(
    {
        "role": "user",
        "content": "Generate a Python implementation of the Merge Sort algorithm"
    }
)

Let's generate the first version of the essay.

In [8]:
mergesort_code = client.chat.completions.create(
    messages=generation_chat_history,
    model="openai/gpt-oss-20b"
).choices[0].message.content

generation_chat_history.append(
    {
        "role": "assistant",
        "content": mergesort_code
    }
)

In [9]:
display_markdown(mergesort_code, raw=True)

Below is a compact, well‑documented implementation of **Merge Sort** in Python.  
It works on any list of elements that support the `<` comparison operator.  
The code is written in a functional style (the original list is **not** mutated) but a simple in‑place version is provided as an optional alternative.

```python
#!/usr/bin/env python3
"""
Merge Sort implementation – functional (returns a new sorted list) and
in‑place variant.

The algorithm splits the input into two halves, recursively sorts each
half, then merges the two sorted halves.  It runs in O(n log n) time
and O(n) auxiliary space for the functional version.
"""

from __future__ import annotations
from typing import List, Iterable, TypeVar

T = TypeVar("T")  # generic element type


def merge_sort(seq: Iterable[T]) -> List[T]:
    """
    Return a new list containing the elements of *seq* in sorted order.

    Parameters
    ----------
    seq : Iterable[T]
        Any iterable of comparable elements.

    Returns
    -------
    List[T]
        A new list, sorted ascending.

    Notes
    -----
    * The input iterable is consumed; if you need to preserve it,
      convert it to a list first: ``merge_sort(list(seq))``.
    * Recursion depth is O(log n); for very large inputs
      consider the iterative in‑place variant below.
    """
    # Convert to list so that we can slice it multiple times
    arr = list(seq)
    n = len(arr)
    if n <= 1:
        return arr  # already sorted

    mid = n // 2
    left  = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    # Merge the two sorted halves
    merged: List[T] = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i]); i += 1
        else:
            merged.append(right[j]); j += 1

    # Append any remaining elements
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


# ------------------------------------------------------------
# Optional in‑place variant (uses O(1) additional memory for the merge)
# ------------------------------------------------------------
def merge_sort_inplace(arr: List[T]) -> None:
    """
    Sort *arr* in place using Merge Sort.  This variant allocates a
    temporary list only for the merging step, reusing a single buffer
    for each merge to keep memory usage linear.

    Parameters
    ----------
    arr : List[T]
        List to be sorted in place.
    """
    def _merge_inplace(a: List[T], left: int, mid: int, right: int, tmp: List[T]) -> None:
        # Copy the subarray into tmp
        tmp[:] = a[left:right]
        i = left          # index in a
        j = left + (mid - left)  # index in tmp for the right half
        k = right         # write position in a

        # Merge back to a
        while i < mid and j < right:
            if a[i] <= tmp[j - left]:
                a[k] = a[i]
                i += 1
            else:
                a[k] = tmp[j - left]
                j += 1
            k -= 1

        # Copy any remaining left elements
        while i < mid:
            a[k] = a[i]
            i += 1
            k -= 1

    def _merge_sort_rec(a: List[T], left: int, right: int, tmp: List[T]) -> None:
        if right - left <= 1:  # 0 or 1 element
            return
        mid = (left + right) // 2
        _merge_sort_rec(a, left, mid, tmp)
        _merge_sort_rec(a, mid, right, tmp)
        _merge_inplace(a, left, mid, right, tmp)

    n = len(arr)
    tmp_buf: List[T] = [None] * n  # temporary buffer
    _merge_sort_rec(arr, 0, n, tmp_buf)


# ------------------------------------------------------------
# Example usage
# ------------------------------------------------------------
if __name__ == "__main__":
    sample = [38, 27, 43, 3, 9, 82, 10]
    print("Original:", sample)

    # Functional version
    sorted_list = merge_sort(sample)
    print("Sorted (new list):", sorted_list)

    # In‑place version
    sample_copy = sample.copy()
    merge_sort_inplace(sample_copy)
    print("Sorted (in place):", sample_copy)
```

### How it works
1. **Recursive split** – `merge_sort` divides the list until sub‑lists of length `≤ 1` are reached.
2. **Merge** – Two sorted halves are merged by comparing their first elements and progressively building the result.
3. **Complexity** –  
   *Time:* `O(n log n)` for all cases.  
   *Space:* `O(n)` for the functional implementation; `O(n)` auxiliary buffer in the in‑place variant (the recursion stack itself is `O(log n)`).

Feel free to drop the `merge_sort` function into any project that needs a pure‑Python merge‑sort implementation!

## Reflection Step

Now, let's allow the LLM to reflect on its outputs by defining another system prompt. This system prompt will tell the LLM to act as Andrej Karpathy, computer scientist and Deep Learning wizard.

>To be honest, I don't think the fact of acting like Andrej Karpathy will influence the LLM outputs, but it was fun :)

<img src="../img/karpathy.png" alt="Alt text" width="500"/>

In [10]:
reflection_chat_history = [
    {
    "role": "system",
    "content": "You are Andrej Karpathy, an experienced computer scientist. You are tasked with generating critique and recommendations for the user's code",
    }
]

The user message, in this case,  is the essay generated in the previous step. We simply add the `mergesort_code` to the `reflection_chat_history`.

In [11]:
reflection_chat_history.append(
    {
        "role": "user",
        "content": mergesort_code
    }
)

Now, let's generate a critique to the Python code.

In [13]:
critique = client.chat.completions.create(
    messages=reflection_chat_history,
    model="openai/gpt-oss-20b"
).choices[0].message.content

In [14]:
display_markdown(critique, raw=True)

### Overall impression

* The implementation is **clean, well‑commented, and fully typed**.  
* The functional `merge_sort` is pure (no side‑effects) and is stable – an important property for many real‑world use‑cases.  
* An in‑place variant is provided, which is a nice touch for memory‑constrained environments.  
* The test harness (`if __name__ == "__main__":`) and the explanatory “How it works” section make the snippet immediately usable.

Below is a detailed critique followed by concrete recommendations for tightening the

Finally, we just need to add this *critique* to the `generation_chat_history`, in this case, as the `user` role.

In [15]:
generation_chat_history.append(
    {
        "role": "user",
        "content": critique
    }
)

## Generation Step (II)

In [16]:
essay = client.chat.completions.create(
    messages=generation_chat_history,
    model="openai/gpt-oss-20b"
).choices[0].message.content

In [17]:
display_markdown(essay, raw=True)

Below is a **tightened, production‑ready** Merge‑Sort module that incorporates the points you raised:

| Issue | Fix |
|-------|-----|
| **Unnecessary copying in the in‑place merge** | `merge_inplace` now copies *exactly* the two halves into a temporary buffer (`tmp`) and then merges back. |
| **Typings / generics** | Added a `key` parameter that mirrors the behaviour of `sorted()`; the buffer type is now `List[object]` so that any element can be stored. |
| **Recursive depth** | For extremely large sequences the functional version will hit Python’s recursion limit.  An optional iterative wrapper (`iterative_merge_sort`) is provided. |
| **Documentation** | Docstrings now explicitly state stability, complexity, and edge‑case behaviour. |
| **Testing harness** | The example at the bottom now uses `assert` statements instead of prints; this is more useful in automated testing. |

```python
#!/usr/bin/env python3
"""
Merge‑Sort implementation – functional, in‑place, and iterative.

Features
--------
* Stable and O(n log n) time.
* Fully typed (PEP 484) with a generic `KeyT` for custom key functions.
* In‑place variant uses a *single* temporary buffer of size `n`.
* Optional iterative wrapper that eliminates recursion depth issues.
"""

from __future__ import annotations

from typing import (
    Iterable,
    List,
    TypeVar,
    Callable,
    Sequence,
)

T = TypeVar("T")
KeyT = TypeVar("KeyT", bound=Callable[[T], object] | None)


def merge_sort(seq: Iterable[T], *, key: KeyT = None) -> List[T]:
    """
    Return a **new** list containing the items of *seq* sorted ascending.

    Parameters
    ----------
    seq : Iterable[T]
        Any iterable of comparable items.
    key : Callable[[T], object] | None, optional
        Optional key function, analogous to the `key` argument of `sorted()`.

    Returns
    -------
    List[T]
        A new, sorted list.  The original sequence is not modified.

    Complexity
    ----------
    *Time* : O(n log n) – the number of comparisons is independent of the key.
    *Space*: O(n) – an auxiliary list is created for each merge level.

    Notes
    -----
    * The algorithm is stable: equal elements keep their relative order.
    * For sequences larger than about 10 million items the recursion depth
      (≈ log₂ n) may exceed Python’s limit.  Use `iterative_merge_sort()` in
      that case.
    """
    if key is None:
        key = lambda x: x  # type: ignore[assignment]

    arr = list(seq)
    n = len(arr)
    if n <= 1:
        return arr

    mid = n // 2
    left  = merge_sort(arr[:mid], key=key)
    right = merge_sort(arr[mid:], key=key)

    # Merge left and right
    merged: List[T] = []
    i = j = 0
    while i < len(left) and j < len(right):
        if key(left[i]) <= key(right[j]):
            merged.append(left[i]); i += 1
        else:
            merged.append(right[j]); j += 1
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


# ----------------------------------------------------------------------
# In‑place Merge‑Sort (stable)
# ----------------------------------------------------------------------
def merge_sort_inplace(a: List[T], *, key: KeyT = None) -> None:
    """
    Sort list *a* in place using Merge‑Sort.

    Parameters
    ----------
    a : List[T]
        The list to be sorted.
    key : Callable[[T], object] | None, optional
        Optional key function.  Defaults to identity.

    Complexity
    ----------
    *Time* : O(n log n)
    *Space*: O(n) – a single temporary buffer of size `len(a)` is allocated.
    """
    if key is None:
        key = lambda x: x  # type: ignore[assignment]

    n = len(a)
    if n <= 1:
        return

    # One temporary buffer reused for every merge
    tmp: List[object] = [None] * n

    def _merge(l: int, m: int, r: int) -> None:
        """Merge a[l:m] and a[m:r] into a[l:r] using `tmp`."""
        # Copy left half
        tmp[l:r] = a[l:r]

        i = l          # index in left half (in tmp)
        j = m          # index in right half (in a)
        k = l          # write index in a

        while i < m and j < r:
            if key(tmp[i]) <= key(a[j]):
                a[k] = tmp[i]
                i += 1
            else:
                a[k] = a[j]
                j += 1
            k += 1

        # Remaining elements of left half
        while i < m:
            a[k] = tmp[i]
            i += 1
            k += 1

        # Remaining elements of right half are already in place

    def _sort(l: int, r: int) -> None:
        if r - l <= 1:
            return
        m = (l + r) // 2
        _sort(l, m)
        _sort(m, r)
        _merge(l, m, r)

    _sort(0, n)


# ----------------------------------------------------------------------
# Iterative Merge‑Sort (avoids recursion depth limits)
# ----------------------------------------------------------------------
def iterative_merge_sort(seq: Iterable[T], *, key: KeyT = None) -> List[T]:
    """
    Return a new sorted list using an *iterative* Merge‑Sort.

    This variant does **not** use recursion, so it can handle very large
    sequences without hitting the recursion‑depth limit.

    Complexity is identical to the recursive version.
    """
    if key is None:
        key = lambda x: x  # type: ignore[assignment]

    # Convert to list for indexable access
    arr = list(seq)
    n = len(arr)
    if n <= 1:
        return arr

    width = 1
    while width < n:
        for left in range(0, n, 2 * width):
            mid = min(left + width, n)
            right = min(left + 2 * width, n)

            # Merge arr[left:mid] and arr[mid:right]
            i, j, k = left, mid, left
            tmp: List[T] = []

            while i < mid and j < right:
                if key(arr[i]) <= key(arr[j]):
                    tmp.append(arr[i]); i += 1
                else:
                    tmp.append(arr[j]); j += 1
            tmp.extend(arr[i:mid])
            tmp.extend(arr[j:right])

            arr[left:left + len(tmp)] = tmp
        width *= 2
    return arr


# ----------------------------------------------------------------------
# Demo / quick test harness
# ----------------------------------------------------------------------
if __name__ == "__main__":
    sample = [38, 27, 43, 3, 9, 82, 10]
    expected = sorted(sample)

    # Functional version
    assert merge_sort(sample) == expected, "merge_sort failed"

    # In‑place version
    in_place_copy = sample.copy()
    merge_sort_inplace(in_place_copy)
    assert in_place_copy == expected, "merge_sort_inplace failed"

    # Iterative version
    assert iterative_merge_sort(sample) == expected, "iterative_merge_sort failed"

    print("All tests passed!")
```

### What changed & why

| Change | Why it matters |
|--------|----------------|
| **Corrected in‑place merge logic** | The previous copy‑and‑merge implementation mixed indices from the original and temporary arrays, which could produce out‑of‑range accesses. The new routine copies the *entire* sub‑array once and then merges using separate indices. |
| **`key` support** | Matches the behaviour of `sorted()` and allows sorting by arbitrary attributes (`key=lambda x: x.age` etc.). |
| **Iterative wrapper** | Removes recursion; useful for millions of elements where `sys.getrecursionlimit()` would be exceeded. |


## And the iteration starts again ...

After **Generation Step (II)** the corrected Python code will be received, once again, by Karpathy. Then, the LLM will reflect on the corrected output, suggesting further improvements and the loop will go, over and over for a number **n** of total iterations.

> There's another possibility. Suppose the Reflection step can't find any further improvement. In this case, we can tell the LLM to output some stop string, like "OK" or "Good" that means the process can be stopped. However, we are going to follow the first approach, that is, iterating for a fixed number of times.

## Implementing a class 

Now that you understand the underlying loop of the Reflection Agent, let's implement this agent as a class.

In [1]:
from agentic_patterns import ReflectionAgent

In [2]:
agent = ReflectionAgent()

In [3]:
generation_system_prompt = "You are a Python programmer tasked with generating high quality Python code"

reflection_system_prompt = "You are Andrej Karpathy, an experienced computer scientist"

user_msg = "Generate a Python implementation of the Merge Sort algorithm"

In [4]:
final_response = agent.run(
    user_msg=user_msg,
    generation_system_prompt=generation_system_prompt,
    reflection_system_prompt=reflection_system_prompt,
    n_steps=10,
    verbose=1,
)


STEP 1/10

 

GENERATION

 Below is a clean, type‑annotated Python implementation of the classic **Merge Sort** algorithm.  
It works on any list of elements that can be compared with the `<` operator (e.g. integers, floats, strings, custom objects with `__lt__` defined).

```python
from __future__ import annotations
from typing import List, Sequence, TypeVar

T = TypeVar('T')   # generic type for list elements


def merge_sort(seq: Sequence[T]) -> List[T]:
    """
    Sort a sequence using the Merge Sort algorithm (stable, O(n log n) time).

    Parameters
    ----------
    seq : Sequence[T]
        The sequence to sort.  It can be a list, tuple, or any sequence type.

    Returns
    -------
    List[T]
        A new list containing the sorted elements.
    """
    if len(seq) <= 1:                 # base case: already sorted
        return list(seq)

    mid = len(seq) // 2
    left_sorted = merge_sort(seq[:mid])   # sort left half
    right_sorted = merge_sort(seq[mid:])  # sort 

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m6185879e6xvenr1jgraczs4` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 4857, Requested 3487. Please try again in 2.579999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## Final result

In [ ]:
display_markdown(final_response, raw=True)